In [1]:
from pyCHX.chx_packages import *
plt.rcParams.update({'figure.max_open_warning': 0})
plt.rcParams.update({ 'image.origin': 'lower'   })
plt.rcParams.update({ 'image.interpolation': 'none'   })
import json
import sys
import dask
sys.path.insert(0, "/nsls2/data/chx/shared/CHX_Software/packages/waxs_auto_processing/")
sys.path.insert(0, "/nsls2/data/chx/shared/CHX_Software/packages/standard_functions/")
from waxs_auto_processing import *
from standard_functions import get_uid_list

import glob
import uuid
import pandas as pd
from pandas.io.formats.style import Styler
from termcolor import colored

%run /nsls2/data/chx/shared/CHX_Software/packages/environment_management/chx_analysis_setup_test_pyCHX.ipynb

pd.set_option('display.max_colwidth', None)

Tiled version 0.1.0a120
/nsls2/conda/envs/2024-2.0-py311-tiled/lib/python3.11/site-packages/databroker/v1.py:72: UserWarning: In databroker 2.x, there are separate notions of 'server' and 'client', and register_handler(...) has no effect on the client. Likely this is being done for you on the server side, so you should not worry about this message unless you encounter trouble loading large array data.
  warnings.warn(


running on: jupyter_hub   environment: standard

setting '_base_path_' as /nsls2/data/chx/legacy/analysis/
setting '_mask_path_' as /nsls2/data/chx/shared/CHX_Setup/Detector_masks/



Tiled version 0.1.0a120


Re-imported pyCHX from /nsls2/data/chx/shared/CHX_Software/packages/pyCHX/
Re-imported chx_compress from /nsls2/data/chx/shared/CHX_Software/packages/pyCHX/chx_compress.py

environment dependent settings and patches:
using "%matplotlib inline" for plotting


/nsls2/conda/envs/2024-2.0-py311-tiled/lib/python3.11/site-packages/databroker/v1.py:72: UserWarning: In databroker 2.x, there are separate notions of 'server' and 'client', and register_handler(...) has no effect on the client. Likely this is being done for you on the server side, so you should not worry about this message unless you encounter trouble loading large array data.
  warnings.warn(


In [2]:
scans = np.arange(149940,149955).tolist() # 
#scans = None # No new scans to add, just load existing database

user = 'lwiegart'
analysts = ['gplautzra','rama'] # additional people who might have processed / analyzed data
cycle = '2024_2'

##############################################################################################################
try: user_=user
except: user_=None
try: cycle_=cycle
except: cycle_=None

[s,uids]=get_uid_list(scans,user=user_,cycle=cycle_,fail_nonexisting=False)

### Load existing database

In [3]:
json_path = _base_path_+'%s/%s/Results/'%(cycle,user)  # don't change
fn='XPCS_WAXS_database' #don't change unless there is a good reason...

col_dict, df_sum = load_collection_database(fn,json_path)

Found existing database /nsls2/data/chx/legacy/analysis/2024_2/lwiegart/Results/XPCS_WAXS_database.json!



Load existing database and append? [y,n]: y


-> loading database...


### Define metadata we want to be able to search for

In [4]:
md_list=['sample','beam position relative to platform','substrate','nozzle','nozzle_diameter [um]','speed [mm_s]','printhead Temp [C]','extrusion speed [ct_s]',
         'eff filament diam [mm]','p1 temperature [C]','p2 temperature [C]','printbed temperature']
accuracy_list=[None, None, None, None, 1, .1, 3, 100,.01,1,1,1]
assert len(md_list) == len(accuracy_list), 'ERROR: length of md_list must match length of  accuracy_list'
print('metadata -> threshold for being identical')
for ii,i in enumerate(md_list):
    print('%s  -> %s'%(i,accuracy_list[ii]))

metadata -> threshold for being identical
sample  -> None
beam position relative to platform  -> None
substrate  -> None
nozzle  -> None
nozzle_diameter [um]  -> 1
speed [mm_s]  -> 0.1
printhead Temp [C]  -> 3
extrusion speed [ct_s]  -> 100
eff filament diam [mm]  -> 0.01
p1 temperature [C]  -> 1
p2 temperature [C]  -> 1
printbed temperature  -> 1


### Add datasets to database

In [5]:
if scans:
    col_dict, df_sum = add_datasets_to_database(uids,col_dict,df_sum,_base_path_,md_list,accuracy_list,remove_duplicates=True)

Datasets to be added to the collection: 
scan_id: 149940   uid: 5cd0a8a4-4d29-4037-ac80-f8119301e757  data type: XPCS    0.25s x 200 fr. Trans:0.1901033750062945 sample: 95wt% HOMO-5wt% PPMA 5wt% Si post 2
scan_id: 149941   uid: b2dea460-550f-4aa3-8a90-dc1e3a643213  data type: WAXS   0.225s x 300 fr
scan_id: 149942   uid: 6aa22819-5bf0-489a-b729-13c253f2b2e2  data type: XPCS    0.005s x 5000 fr. Trans:1.0 sample: 95wt% HOMO-5wt% PPMA 5wt% Si in-situ series
scan_id: 149943   uid: 0aa80609-13e9-4cb2-b1a5-71bad54eb2ec  data type: WAXS   0.045s x 800 fr
scan_id: 149944   uid: 32278e28-896c-4e35-89a0-90a8ab0dc38d  data type: XPCS    0.005s x 5000 fr. Trans:1.0 sample: 95wt% HOMO-5wt% PPMA 5wt% Si in-situ series
scan_id: 149945   uid: 8d36138a-6ee7-4d1f-99c2-845214f718df  data type: XPCS    0.25s x 200 fr. Trans:0.1901033750062945 sample: 95wt% HOMO-5wt% PPMA 5wt% Si post 1
scan_id: 149946   uid: c18ce63e-821f-445e-a91e-8f8d47b3503c  data type: XPCS    0.25s x 200 fr. Trans:0.190103375006294

### Check for updates on data processing and analysis for user and analysts

In [6]:
# get all usernames for which we want to check whether they did data processing/analysis
name_list = [user]+analysts
# get all uids in col_dict:
all_uids=[] # can also provide a manual list of specific uids...
for cu in col_dict.keys():
    for d in ['XPCS','WAXS']:
        all_uids+=list(col_dict[cu]['data'][d].keys())

In [7]:
col_dict,df_sum = update_processing_analysis(all_uids,col_dict,df_sum,name_list,_base_path_)

Updating uids in database…: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 13/13 [00:00<00:00, 30.94it/s]


### Show current database entries

In [8]:
styled_df(df_sum)

,col_uid,XPCS data,WAXS data,sample,beam position relative to platform,substrate,nozzle,nozzle_diameter [um],speed [mm_s],printhead Temp [C],extrusion speed [ct_s],eff filament diam [mm],p1 temperature [C],p2 temperature [C],printbed temperature,md error,XPCS processed,XPCS exit status,WAXS processed,WAXS exit status,timing
0,8ddf165f383741a3b3105db6a7232bc4,149942,,95wt% HOMO-5wt% PPMA 5wt% Si,0.250,Kapton tape + magigoo PP,hyrel3D FFF,740,10.0,234.0,1117.2,1.0,34.2,120.1,120.1,False,False,False,False,True,OK
1,fad3419e419442fba688ab48d6a0c103,"149944, 149945, 149946","149943, 149947",95wt% HOMO-5wt% PPMA 5wt% Si,0.070,Kapton tape + magigoo PP,hyrel3D FFF,740,10.0,233.0,1117.2,1.0,34.3,120.0,120.0,False,True,True,True,True,WARNING
2,7ee835ab39444657850d5dc392e14cf4,"149949, 149950, 149951","149948, 149952",95wt% HOMO-5wt% PPMA 5wt% Si,0.070,Kapton tape + magigoo PP,hyrel3D FFF,740,10.0,234.0,1117.2,1.0,34.1,120.0,120.0,False,True,True,False,True,OK
3,48c0540dcecc4d91bf97af940dff762c,149954,149953,95wt% HOMO-5wt% PPMA 5wt% Si,0.070,Kapton tape + magigoo PP,hyrel3D FFF,740,10.0,234.0,1117.2,1.0,31.2,100.1,100.1,False,False,False,False,True,OK


### SAVE Collection dict AND pandas_dict as one json file:

In [ ]:
save_collection_database(fn, json_path, col_dict, df_sum)

#### Some ways to search the database

In [ ]:
# # filter results by multiple conditions

# filt = {"XPCS exit status":True,"md error":False}

# success=df_sum
# for k in filt.keys():
#     success = success[success[k]==filt[k]]
# styled_df(success.head())


# df_sum['printbed temperature'].between(100,120,inclusive='both')

### Try to search for something in a somewhat human understandable way:

In [ ]:
search="(('HOMO' AND '95') OR 'PPMA') in 'sample' & 'magigoo' in 'substrate' & 110 < 'printbed temperature' <=120 & ('OK' OR 'ok-ish') in 'timing'  & 'md error' == False"


search_result=search_pd_database(df_sum,search,verbose=False)

#### look at the search results:

In [ ]:
# show the sub-set that matches the search criteria
display(styled_df(df_sum[search_result['index mask']]))

print('collection uids matching the search criteria: ',search_result['filtered collection uids'])

In [ ]:
# How to get the col_uid back for a known single scan_id:

df_found=df_sum[df_sum['XPCS data'].str.contains('149949')]
col_uid = np.array(df_found['col_uid'].values,dtype=str)
if len(col_uid)==1:
    col_uid=col_uid[0]
print(col_uid)

### Run a detailed report on a collection of datasets

In [ ]:
report_df=collection_report(col_uid,col_dict,df_sum,md_list,accuracy_list,timelineplot=True,show_oavs=True,plot_dict = {'y_extend':[500,500],'cross':[441,2448-1492]}) # should define cross position better in md

In [ ]:
_the_END

### can we save the report?

In [ ]:
import dataframe_image as dfi

In [ ]:
dfi.export(report_df.style.bar( color='#5e81f2', vmax=100, vmin=0).set_table_attributes('style="font-size: 17px"').set_properties(**{'color': 'black !important','border': '1px black solid !important'}).set_table_styles([{
    'selector': 'th','props': [('border', '1px black solid !important')]}]), 'pandas_report_test.png',table_conversion='matplolib') #.style.apply(lambda _: mask.map(color_warning_report), axis=None).format(precision=3)